# IndicXlit baseline benchmark

Runs AI4Bharat's published IndicXlit model on the same three test sets as our own models, with identical normalization, so the comparison table is apples-to-apples.

**Settings**: no GPU needed (leave Accelerator = None). **Internet = On** required.

**Why the Python 3.10 sandbox below**: IndicXlit depends on fairseq, which has been unmaintained since ~2022 and no longer builds on Python 3.11+ (Kaggle's image is 3.12, Windows fails too). So we create an isolated Python 3.10 environment with `uv` and an older pip that tolerates fairseq's dated dependency metadata, and run only the benchmark inside it.

In [ ]:
!git clone https://github.com/Akash-5675/Manglish---Malayalam-Transliteration.git repo
%cd repo

## Rebuild the test sets (system Python — deterministic, same seed 42)

In [ ]:
!python src/data/download.py
!python src/data/preprocess.py
!python src/data/split.py

## Python 3.10 sandbox for fairseq/IndicXlit

In [ ]:
!pip install -q uv
!uv venv --python 3.10 /kaggle/venv310
!uv pip install --python /kaggle/venv310/bin/python "pip<24.1"
# CPU-only torch first so fairseq doesn't drag in the multi-GB CUDA build
!/kaggle/venv310/bin/python -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!/kaggle/venv310/bin/python -m pip install -q ai4bharat-transliteration jiwer
!/kaggle/venv310/bin/python -c "from ai4bharat.transliteration import XlitEngine; print('IndicXlit import OK')"

## Run IndicXlit on 2000-pair samples of each test set

In [ ]:
!/kaggle/venv310/bin/python src/evaluation/benchmark_indicxlit.py --subset 2000

In [ ]:
!cp logs/indicxlit_results.json /kaggle/working/indicxlit_results.json
print("done -- download indicxlit_results.json from the Output tab")